In [1]:
import torch
import numpy as np
import pandas as pd
from PIL import Image
from pathlib import Path
from typing import List, Dict, Tuple, Optional
from transformers import CLIPModel, CLIPProcessor
from tqdm import tqdm
import json

In [ ]:
def extract_clip_embeddings(image_folder: str = 'posters',clip_model: str = 'openai/clip-vit-base-patch32',batch_size: int = 32,device: str = None) -> Tuple[np.ndarray, List[Dict]]:
    if device is None:
        device = 'cuda' if torch.cuda.is_available() else 'cpu'
    
    model = CLIPModel.from_pretrained(clip_model).to(device)
    processor = CLIPProcessor.from_pretrained(clip_model)
    model.eval()
    
    img_folder = Path(image_folder)
    image_files = []
    for ext in ['*.jpg', '*.jpeg', '*.png']:
        image_files.extend(list(img_folder.glob(ext)))
    
    metadata = []
    for img_path in image_files:
        metadata.append({
            'image_path': str(img_path),
            'title': img_path.stem.replace('_', ' ')
        })
    
    all_embeddings = []
    with torch.no_grad():
        for i in tqdm(range(0, len(metadata), batch_size)):
            batch_data = metadata[i:i+batch_size]
            images = [Image.open(item['image_path']).convert('RGB') for item in batch_data]
            
            inputs = processor(images=images, return_tensors="pt", padding=True).to(device)
            image_features = model.get_image_features(**inputs)
            image_features = image_features / image_features.norm(dim=-1, keepdim=True)
            
            all_embeddings.append(image_features.cpu().numpy())
    
    embeddings = np.vstack(all_embeddings)
    return embeddings, metadata

In [3]:
def save_to_vector_db(
    embeddings: np.ndarray,
    metadata: List[Dict],
    db_type: str = 'chromadb',
    db_path: str = './vector_db',
    collection_name: str = 'movie_posters'
) -> Dict:
    
    db_info = {
        'db_type': db_type,
        'db_path': db_path,
        'collection_name': collection_name
    }
    
    if db_type == 'chromadb':
        import chromadb
        from chromadb.config import Settings
        
        client = chromadb.Client(Settings(persist_directory=db_path, anonymized_telemetry=False))
        
        try:
            client.delete_collection(name=collection_name)
        except:
            pass
        
        collection = client.create_collection(name=collection_name)
        
        collection.add(
            ids=[str(i) for i in range(len(embeddings))],
            embeddings=embeddings.tolist(),
            documents=[m['title'] for m in metadata],
            metadatas=[{'title': m['title'], 'image_path': m['image_path']} for m in metadata]
        )
        
    elif db_type == 'faiss':
        import faiss
        
        Path(db_path).mkdir(parents=True, exist_ok=True)
        embeddings_normalized = embeddings / np.linalg.norm(embeddings, axis=1, keepdims=True)
        
        index = faiss.IndexFlatIP(embeddings.shape[1])
        index.add(embeddings_normalized.astype('float32'))
        
        index_path = Path(db_path) / f'{collection_name}.index'
        faiss.write_index(index, str(index_path))
        
        db_info['index_path'] = str(index_path)
        
    elif db_type == 'numpy':
        Path(db_path).mkdir(parents=True, exist_ok=True)
        np.save(Path(db_path) / f'{collection_name}_embeddings.npy', embeddings)
        db_info['embeddings_path'] = str(Path(db_path) / f'{collection_name}_embeddings.npy')
    
    with open(Path(db_path) / 'metadata.json', 'w') as f:
        json.dump(metadata, f)
    
    with open(Path(db_path) / 'db_info.json', 'w') as f:
        json.dump(db_info, f)
    
    return db_info

In [4]:
def find_similar_movies(
    query: str or int,
    db_path: str = './vector_db',
    top_k: int = 10
) -> List[Dict]:
    
    with open(Path(db_path) / 'db_info.json', 'r') as f:
        db_info = json.load(f)
    
    with open(Path(db_path) / 'metadata.json', 'r') as f:
        metadata = json.load(f)
    
    if isinstance(query, str):
        query_idx = next((i for i, m in enumerate(metadata) if query.lower() in m['title'].lower()), None)
        if query_idx is None:
            raise ValueError(f"Movie '{query}' not found")
    else:
        query_idx = query
    
    db_type = db_info['db_type']
    
    if db_type == 'chromadb':
        import chromadb
        from chromadb.config import Settings
        
        client = chromadb.Client(Settings(persist_directory=db_path, anonymized_telemetry=False))
        collection = client.get_collection(name=db_info['collection_name'])
        
        query_result = collection.get(ids=[str(query_idx)], include=['embeddings'])
        results = collection.query(query_embeddings=query_result['embeddings'], n_results=top_k + 1)
        
        similar = []
        for i, idx_str in enumerate(results['ids'][0]):
            idx = int(idx_str)
            if idx != query_idx:
                similar.append({
                    'index': idx,
                    'title': metadata[idx]['title'],
                    'similarity': 1 - results['distances'][0][i],
                    'image_path': metadata[idx]['image_path']
                })
        
        return similar[:top_k]
    
    elif db_type == 'faiss':
        import faiss
        
        index = faiss.read_index(db_info['index_path'])
        embeddings = np.zeros((index.ntotal, index.d), dtype='float32')
        index.reconstruct_n(0, index.ntotal, embeddings)
        
        query_normalized = embeddings[query_idx:query_idx+1]
        distances, indices = index.search(query_normalized, top_k + 1)
        
        similar = []
        for idx, dist in zip(indices[0], distances[0]):
            if idx != query_idx:
                similar.append({
                    'index': int(idx),
                    'title': metadata[idx]['title'],
                    'similarity': float(dist),
                    'image_path': metadata[idx]['image_path']
                })
        
        return similar[:top_k]
    
    elif db_type == 'numpy':
        embeddings = np.load(db_info['embeddings_path'])
        query_embedding = embeddings[query_idx:query_idx+1]
        similarities = np.dot(embeddings, query_embedding.T).flatten()
        sorted_indices = np.argsort(similarities)[::-1]
        
        similar = []
        for idx in sorted_indices:
            if idx != query_idx and len(similar) < top_k:
                similar.append({
                    'index': int(idx),
                    'title': metadata[idx]['title'],
                    'similarity': float(similarities[idx]),
                    'image_path': metadata[idx]['image_path']
                })
        
        return similar

In [ ]:
def get_movie_image(movie_id: str or int,db_path: str = './vector_db') -> Image.Image:
    
    with open(Path(db_path) / 'metadata.json', 'r') as f:
        metadata = json.load(f)
    
    if isinstance(movie_id, str):
        movie = next((m for m in metadata if movie_id.lower() in m['title'].lower()), None)
        if movie is None:
            raise ValueError(f"Movie '{movie_id}' not found")
    else:
        movie = metadata[movie_id]
    
    return Image.open(movie['image_path'])

In [ ]:
if __name__ == "__main__":
    embeddings, metadata = extract_clip_embeddings('posters')
    db_info = save_to_vector_db(embeddings, metadata, db_type='chromadb')
    results = find_similar_movies("Inception", top_k=5)
    
    for i, movie in enumerate(results, 1):
        print(f"{i}. {movie['title']} - Similarity: {movie['similarity']:.4f}")
    
    img = get_movie_image("Inception")
    print(f"Image size: {img.size}")